In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import re


In [ ]:
# find all csv files
files = sorted(glob.glob("../csv/raw_data/capacity_cars_*_runs_48_ticks_600.csv"))
print("Found files:", len(files))
files

Found files: 16


['../csv/raw_data\\capacity_cars_100_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_110_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_120_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_130_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_140_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_150_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_160_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_170_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_180_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_190_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_200_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_210_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_220_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_230_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_80_runs_48_ticks_600.csv',
 '../csv/raw_data\\capacity_cars_90_runs_48_ticks_600.csv']

In [ ]:
# combined all csv here
all_dfs = []

for sf in files:
    # parse total cars from filename: capacity_cars_150_runs_20_ticks_1000.csv -> 150
    m = re.search(r"capacity_cars_(\d+)_", sf)
    total_cars = int(m.group(1))

    df = pd.read_csv(sf)
    df["total_cars"] = total_cars
    df["source_file"] = sf
    all_dfs.append(df)

combined_csv = pd.concat(all_dfs, ignore_index=True)

print("Combined shape:", combined_csv.shape)
#combined_csv.dtypes
combined_csv

Combined shape: (7680, 5)


,run,road_id,flow,total_cars,source_file
0,1,"(0,4)",69,100,../csv/raw_data\capacity_cars_100_runs_48_tick...
1,1,"(0,3)",70,100,../csv/raw_data\capacity_cars_100_runs_48_tick...
2,1,"(0,2)",73,100,../csv/raw_data\capacity_cars_100_runs_48_tick...
3,1,"(0,1)",59,100,../csv/raw_data\capacity_cars_100_runs_48_tick...
4,1,"(0,0)",83,100,../csv/raw_data\capacity_cars_100_runs_48_tick...
...,...,...,...,...,...
7675,48,"(1,0)",40,90,../csv/raw_data\capacity_cars_90_runs_48_ticks...
7676,48,"(1,1)",74,90,../csv/raw_data\capacity_cars_90_runs_48_ticks...
7677,48,"(1,2)",60,90,../csv/raw_data\capacity_cars_90_runs_48_ticks...
7678,48,"(1,3)",53,90,../csv/raw_data\capacity_cars_90_runs_48_ticks...


In [ ]:
# Tidy data
combined_csv["flow"] = (
    combined_csv["flow"]
    .astype(str)
)
combined_csv["flow"] = pd.to_numeric(combined_csv["flow"])

combined_csv["flow"].dtype


dtype('int64')

In [ ]:
# find the capacity for every road
capacity_per_road = (
    combined_csv
    .groupby("road_id", as_index=False)["flow"]
    .max()
    .rename(columns={"flow": "max_flow"})
)

capacity_per_road["flow_per_tick"] = (
    np.ceil((capacity_per_road["max_flow"] / 600) * 1000) / 1000
)

capacity_per_road
capacity_per_road


,road_id,max_flow,flow_per_tick
0,"(0,0)",94,0.157
1,"(0,1)",96,0.160
2,"(0,2)",96,0.160
3,"(0,3)",97,0.162
4,"(0,4)",95,0.159
5,"(1,0)",94,0.157
6,"(1,1)",93,0.155
7,"(1,2)",95,0.159
8,"(1,3)",97,0.162
9,"(1,4)",95,0.159


In [ ]:
# export the csv
capacity_per_road.to_csv("../csv/explored_data/capacity_per_road.csv", index=False)